# 06 · The arranger: matching the slider mix

The finder hands over 500 candidates (100 per bucket). The arranger picks the **20 you actually see**, so they match your sliders while staying relevant.

**How it works**

1. **Percentages → slots.** 20 articles × your mix. E.g. 30/20/20/20/10 → 6 News, 4 Learn, 4 Lifestyle, 4 Entertain, 2 Sports (rounded so it always adds up to 20).
2. **Pick one article at a time.** Each pick maximizes: `relevance + strictness × (open slots left in that article's bucket)`.
3. **Order the page** by relevance, best first.

**The strictness knob (λ)** decides how much the sliders matter:
- **0:** ignore the sliders, plain top 20 (a normal feed).
- **High:** fill the slots exactly.

**This notebook:** sweep strictness from 0 to 1, look at the trade-off (Chart 1), then pick one default value with a clear rule.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.arranger import PAGE_SIZE, arrange, mix_to_slots, normalize_scores
from src.data import BUCKETS, explode_impressions, load_behaviors, load_news, split_by_time
from src.finder import hourly_clicks, load_embeddings
from src.metrics import mix_error, relevance_kept
from src.two_tower import MODEL_PATH, TwoTower, device, find_candidates, history_matrix, popularity_table

dev = device()

## Get the finder's 500 candidates for every Nov 14 page view

Uses the trained model from notebook 05 (`artifacts/two_tower.pt`): similarity + 0.05 × trending, top 100 per bucket.

In [ ]:
news = load_news()
news_ids, vectors = load_embeddings()
index_of = {n: i for i, n in enumerate(news_ids)}
bucket_of = news.dropna(subset=["bucket"]).set_index("news_id").bucket.to_dict()

behaviors = load_behaviors()
train_b, val_b = split_by_time(behaviors)
row_of, hist = history_matrix(behaviors, index_of)
article_vectors = torch.as_tensor(vectors, device=dev)
history = torch.as_tensor(hist, device=dev)

model = TwoTower().to(dev)
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

val_shown = explode_impressions(val_b)
pool = np.array(sorted(set(val_shown.news_id) & set(bucket_of)))
pool_idx = torch.as_tensor([index_of[n] for n in pool], device=dev)
pool_buckets = np.array([bucket_of[n] for n in pool], dtype=object)
pool_bucket_idx = torch.as_tensor([BUCKETS.index(b) for b in pool_buckets], device=dev)

val_imp = (
    val_shown[(val_shown.clicked == 1) & val_shown.news_id.isin(bucket_of)]
    .groupby(["impression_id", "user_id", "time"]).news_id.agg(list).reset_index()
)
val_hours = val_imp.time.dt.floor("h").values
hour_row, table = popularity_table(hourly_clicks(behaviors), val_hours, index_of)

cols, scores = find_candidates(
    model,
    user_rows=torch.as_tensor([row_of[u] for u in val_imp.user_id], device=dev),
    hour_rows=torch.as_tensor([hour_row[h] for h in pd.to_datetime(val_hours)], device=dev),
    history=history, article_vectors=article_vectors, popularity=torch.as_tensor(table, device=dev),
    pool_idx=pool_idx, pool_buckets=pool_buckets, buckets=BUCKETS,
)
relevance = normalize_scores(scores)       # (page views, 500), best candidate = 1
candidate_buckets = pool_bucket_idx[cols]  # bucket index of each candidate
print(f"{cols.shape[0]:,} page views × {cols.shape[1]} candidates")

## How we measure a page

- **Mix error:** how far the page's mix is from the target, in % (0 = perfect). Asking for 10% Sports and getting 25% is 15 points off.
- **Articles off:** the same idea counted in articles: how many of the 20 sit in the wrong bucket (0 = exact match).
- **Pages exact:** share of pages with 0 articles off.
- **Relevance kept:** total relevance of the 20 picked vs. the plain top 20 (100% = nothing lost). Relevance is the finder's score rescaled per user so the best candidate is 1 and the worst is 0.
- **Recall@20:** share of the user's real clicks that land in the 20. Caveat: those clicks happened on MSN's normal feed, so this is biased toward the normal mix.

In [ ]:
# Each page view's real clicks as pool positions (padded with -1), for Recall@20
position_of = {n: i for i, n in enumerate(pool)}
clicked = [[position_of[n] for n in c] for c in val_imp.news_id]
clicked_t = torch.full((len(clicked), max(map(len, clicked))), -1, device=dev)
for i, c in enumerate(clicked):
    clicked_t[i, : len(c)] = torch.as_tensor(c)

top_relevance = relevance[:, :PAGE_SIZE].cpu().numpy()  # the plain top 20

def run(targets, strictness):
    """Arrange every page view for the given target mixes and return the metrics (averaged over page views)."""
    slots = mix_to_slots(targets)
    chosen = arrange(relevance, candidate_buckets, torch.as_tensor(slots, device=dev), strictness)
    page_buckets = candidate_buckets.gather(1, chosen).cpu().numpy()
    counts = np.stack([(page_buckets == b).sum(1) for b in range(len(BUCKETS))], axis=1)
    off = np.abs(counts - slots).sum(1) / 2  # each misplaced article counts once
    page = cols.gather(1, chosen)
    hits = (page[:, :, None] == clicked_t[:, None, :]).any(1).sum(1) / (clicked_t >= 0).sum(1)
    return {
        "mix_error_%": 100 * mix_error(page_buckets, targets).mean(),
        "articles_off": off.mean(),
        "pages_exact_%": 100 * (off == 0).mean(),
        "relevance_kept_%": 100 * relevance_kept(relevance.gather(1, chosen).cpu().numpy(), top_relevance).mean(),
        "recall@20": hits.mean().item(),
    }

## Sweep the strictness knob

Four slider settings:
- **Balanced:** 20 / 20 / 20 / 20 / 20
- **Less news:** 10 / 30 / 20 / 20 / 20
- **Learner:** 10 / 50 / 20 / 10 / 10
- **Random:** a different random mix for every page view (shows it works for any slider setting)

Each is run at 15 strictness values from 0 to 1.

In [ ]:
n = len(val_imp)
rng = np.random.default_rng(0)
LEARNER = [0.1, 0.5, 0.2, 0.1, 0.1]
settings = {
    "Balanced": np.tile([0.2, 0.2, 0.2, 0.2, 0.2], (n, 1)),
    "Less news": np.tile([0.1, 0.3, 0.2, 0.2, 0.2], (n, 1)),
    "Learner": np.tile(LEARNER, (n, 1)),
    "Random": rng.dirichlet(np.ones(len(BUCKETS)), n),
}
STRICTNESS = [0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.6, 0.7, 0.85, 1.0]

sweep = pd.DataFrame([
    {"setting": name, "strictness": s, **run(targets, s)}
    for name, targets in settings.items() for s in STRICTNESS
])
average = sweep.groupby("strictness")[["mix_error_%", "articles_off", "pages_exact_%", "relevance_kept_%", "recall@20"]].mean()
average.round(2)

## Chart 1: following the sliders keeps ~90% of relevance

Each line is one slider setting; each dot is a strictness level, from 0 (left: normal feed, sliders ignored) to 1 (right: mix followed exactly).

In [ ]:
COLORS = {"Balanced": "#2a78d6", "Less news": "#eb6834", "Learner": "#1baf7a", "Random": "#eda100"}
LABEL_OFFSET = {"Balanced": (8, 0, "left"), "Less news": (8, 0, "left"), "Learner": (8, 0, "left"), "Random": (-8, -12, "right")}

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for name, group in sweep.groupby("setting", sort=False):
    ax.plot(group["mix_error_%"], group["relevance_kept_%"], color=COLORS[name], lw=2, marker="o", ms=6,
            markeredgecolor="white", markeredgewidth=1.2, label=name)
    end = group.iloc[-1]
    dx, dy, ha = LABEL_OFFSET[name]
    ax.annotate(f"{name}  {end['relevance_kept_%']:.0f}%", (end["mix_error_%"], end["relevance_kept_%"]),
                xytext=(dx, dy), textcoords="offset points", ha=ha, va="center", fontsize=9, color="#333333")

ax.set_xlabel("Mix error: distance from the slider mix (%)   closer to what you asked for →")
ax.set_ylabel("Relevance kept (%)")
ax.set_title("Following the sliders exactly keeps ~90% of relevance", loc="left", fontsize=12)
ax.invert_xaxis()
ax.set_ylim(85, 101.5)
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower left")
ax.annotate("normal feed (sliders ignored)", (sweep["mix_error_%"].max(), 100), xytext=(-4, 10),
            textcoords="offset points", ha="left", fontsize=9, color="#666666")
fig.tight_layout()
fig.savefig("../results/figures/chart1_relevance_vs_mix.png", dpi=150)
plt.show()

## Choosing the default strictness

Chart 1 shows the trade-off, but the app needs **one default**.

**The rule:** pick the *smallest* strictness where **at least 90% of pages match the sliders exactly** (averaged over the four slider settings).

Why this rule:
- People judge the sliders by counting. If you set 50% Learn and see 9 Learn articles instead of 10, the slider feels broken. So matching exactly on almost every page matters most.
- Taking the *smallest* such value means we don't give up relevance for strictness we don't need.

In [ ]:
DEFAULT_STRICTNESS = average.index[average["pages_exact_%"] >= 90][0]
chosen_row = average.loc[DEFAULT_STRICTNESS]
print(f"Chosen strictness: {DEFAULT_STRICTNESS}")
print(f"  pages matching exactly: {chosen_row['pages_exact_%']:.0f}%")
print(f"  articles off per page:  {chosen_row['articles_off']:.2f}")
print(f"  relevance kept:         {chosen_row['relevance_kept_%']:.1f}%")

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
for name, group in sweep.groupby("setting", sort=False):
    style = dict(color=COLORS[name], lw=2, marker="o", ms=5, markeredgecolor="white", markeredgewidth=1, label=name)
    left.plot(group["strictness"], group["pages_exact_%"], **style)
    right.plot(group["strictness"], group["relevance_kept_%"], **style)

for ax in (left, right):
    ax.axvline(DEFAULT_STRICTNESS, color="#555555", ls="--", lw=1)
    ax.set_xlabel("Strictness (λ)")
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)

left.axhline(90, color="#999999", ls=":", lw=1)
left.text(0.98, 91.5, "90% of pages exact", ha="right", fontsize=8, color="#666666", transform=left.get_yaxis_transform())
left.set_ylabel("Pages that match the sliders exactly (%)")
left.set_title("Exactness climbs fast, then flattens", loc="left", fontsize=11)
left.annotate(f"chosen: {DEFAULT_STRICTNESS}", (DEFAULT_STRICTNESS, 45), xytext=(6, 0), textcoords="offset points", fontsize=9)
left.legend(frameon=False, loc="lower right")

right.set_ylabel("Relevance kept (%)")
right.set_title("Relevance cost of being strict", loc="left", fontsize=11)
right.annotate(f"chosen: {DEFAULT_STRICTNESS}\naverage kept: {chosen_row['relevance_kept_%']:.1f}%", (DEFAULT_STRICTNESS, 98.5),
               xytext=(6, 0), textcoords="offset points", fontsize=9, va="center")

fig.tight_layout()
fig.savefig("../results/figures/choosing_strictness.png", dpi=150)
plt.show()

### Why 0.5 and not its neighbors

| Strictness | Pages exact | Articles off | Relevance kept | Verdict |
|---|---|---|---|---|
| 0.1 | ~1% | ~2.1 | ~94% | Most pages are 2 articles off: sliders feel loose |
| 0.3 | ~34% | ~0.7 | ~92% | Two in three pages still off by about one article |
| 0.4 | ~72% | ~0.3 | ~91% | Close, but more than 1 in 4 pages is still off |
| **0.5** | **~94%** | **~0.06** | **~90%** | **First value past 90% exact: sliders work, small cost** |
| 0.7 – 1.0 | ~100% | 0 | ~90% | No better: exactness has already maxed out |

- Random slider mixes are the hardest case: at 0.5 they match exactly on 84% of pages (vs 97–99% for the three personas) and pass 90% at 0.6. The relevance difference between 0.5 and 0.6 is only about 0.2%, so either would be fine. We keep the rule's answer.
- Chosen on the validation day (Nov 14).

## Example: one user, normal feed vs. a "Learner" mix at strictness 0.5

Learner = 10% News, 50% Learn, 20% Lifestyle, 10% Entertain, 10% Sports.

In [ ]:
user = 0
slots = torch.as_tensor(mix_to_slots(np.array([LEARNER])), device=dev)

def page_for(strictness):
    chosen = arrange(relevance[user : user + 1], candidate_buckets[user : user + 1], slots, strictness)[0]
    ids = pool[cols[user, chosen].cpu().numpy()]
    return news.set_index("news_id").loc[ids, ["bucket", "title"]].reset_index(drop=True)

normal, learner = page_for(0), page_for(DEFAULT_STRICTNESS)
print("Bucket counts:")
print(pd.DataFrame({
    "target": mix_to_slots(np.array([LEARNER]))[0],
    "normal feed": normal.bucket.value_counts().reindex(BUCKETS).fillna(0).astype(int).values,
    f"learner mix (λ={DEFAULT_STRICTNESS})": learner.bucket.value_counts().reindex(BUCKETS).fillna(0).astype(int).values,
}, index=BUCKETS))
learner.head(10)

## Takeaways

- **A normal feed ignores your sliders.** With strictness 0 the page is 37–54 points away from the requested mix; it just follows whatever scores highest (mostly News and Entertain).
- **Following the sliders exactly keeps 88–93% of relevance.** The further your mix is from your natural taste, the more it costs: Balanced keeps 93%, Learner (50% Learn) keeps 88%.
- **Default strictness = 0.5.** It's the smallest value where at least 90% of pages match the sliders exactly (94% of pages, 0.06 articles off on average), at about 90% relevance kept. Stricter values add nothing.
- **It works for any slider setting.** Random mixes behave like the personas; their mix error can't reach exactly 0 only because 20 articles can't express every percentage (each article is 5%).
- **Real clicks drop (Recall@20 0.22 → about 0.15 at 0.5).** Expected and partly unfair: those clicks were made on MSN's normal feed, so any different mix looks worse on this measure. This is exactly the stated-vs-revealed preference question the balancer (Week 4) deals with.

In [ ]:
# Save the full sweep for the README
sweep.round(4).to_csv("../results/arranger_sweep.csv", index=False)